In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.lines import Line2D
import os

# --------------------------
# 1. Config & Data Loading
# --------------------------
base_dir = "/Users/ewellmeyer/Documents/research/weights/"
channels = [2, 4, 8, 16, 32, 64, 128]

def get_gn(ch):
    if ch in [1]: return 1
    if ch in [2]: return 2
    if ch in [4]: return 4
    if ch in [6]: return 6
    return 8

def load_run_data(run_type_suffix=""):
    results = {
        'softmax': {'hadgem': []},
        'baseline_simple': {'hadgem': None},
    }

    for ch in channels:
        gn = get_gn(ch)
        path_sm = f"{base_dir}ens_HG789_PR_dPdK_Softmax_unet6SR_1x_ch{ch}_k3_128x_dPbins64_gn{gn}{run_type_suffix}/"

        try:
            d_sm = np.load(f"{path_sm}softmax_ensemble_analysis_arrays.npz")
            idx = d_sm['test_indices']
            results['softmax']['hadgem'].append(np.median(d_sm['rmse_softmax_mean'][idx]))
            if results['baseline_simple']['hadgem'] is None:
                results['baseline_simple']['hadgem'] = np.median(d_sm['rmse_ppe'][idx])
        except FileNotFoundError:
            results['softmax']['hadgem'].append(np.nan)
            print(f"Missing Softmax files for Ch={ch}, Suffix='{run_type_suffix}'")

    return results

# Load Data
print("Loading Standard Runs...")
data_std = load_run_data(run_type_suffix="")

# --------------------------
# 2. Plotting
# --------------------------
sns.set_context("paper", font_scale=1.4)
sns.set_style("ticks")

fig, ax = plt.subplots(figsize=(8, 6))

c_soft = '#1f77b4'

# Baseline
if data_std['baseline_simple']['hadgem'] is not None:
    b_hg = data_std['baseline_simple']['hadgem']
    ax.axhline(b_hg, color='gray', linestyle=':', linewidth=1.5, alpha=0.8, zorder=1)
    ax.text(channels[0], b_hg, 'PPE mean - PPE Test', color='gray', fontsize=9,
            va='bottom', ha='left', fontweight='bold')

# Softmax HadGEM
ax.plot(channels, data_std['softmax']['hadgem'], color=c_soft, linestyle='-',
        marker='o', lw=2, ms=6, label='Base Model')

ax.set_xlabel("Network Channels per Layer", fontsize=12)
ax.set_ylabel(r"Median RMSE (mm yr$^{-1}$ K$^{-1}$)", fontsize=12, fontweight='bold')
ax.set_xscale('log', base=2)
ax.set_xticks(channels)
ax.set_xticklabels(channels)
ax.set_ylim(35, 90)
ax.set_xlim(1.5, 135)
ax.grid(True, which='major', linestyle='--', alpha=0.4)
sns.despine(ax=ax)

legend_elements = [
    Line2D([0], [0], color=c_soft, lw=3, label='Base Model (HadGEM Test)'),
    Line2D([0], [0], color='gray', linestyle=':', lw=2, label='HadGEM Baseline (PPE Train mean)'),
]
ax.legend(handles=legend_elements, frameon=False, fontsize=11)

plt.tight_layout()
plt.show()